# 05 — Stockage NoSQL avec MongoDB

## Objectif du notebook

Ce notebook met en place la partie **Architecture de Stockage Big Data & NoSQL** du projet Web Mining.

Nous allons stocker deux datasets dans MongoDB :

1. `final_data_ai_jobs_clean_reduced_2020_2026.csv`  
   → collection `jobs_clean`  
   → une ligne = une offre d'emploi.

2. `final_data_ai_jobs_skills_exploded.csv`  
   → collection `skills_exploded`  
   → une ligne = une compétence liée à une offre.

La base MongoDB créée sera :

```text
data_ai_jobs_db
```

Collections :

```text
jobs_clean
skills_exploded
```

MongoDB est utilisé parce que les offres d’emploi sont des données semi-structurées : titre, pays, date, compétences, source, remote status, salaire, etc.

## 1. Importation des bibliothèques

In [2]:
pip install pymongo

  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   --------

In [3]:
import os
import pandas as pd
import numpy as np
import warnings
from pymongo import MongoClient

warnings.filterwarnings("ignore")

## 2. Définir le dossier de travail

Les grands fichiers de données ne sont pas versionnés dans GitHub.

Par défaut, ce notebook cherche les fichiers dans le dossier `data/` du dépôt.
Vous pouvez aussi définir la variable d'environnement `WEBMINING_DATA_DIR`
pour utiliser un autre dossier local.


In [4]:
from pathlib import Path

configured_data_dir = os.getenv("WEBMINING_DATA_DIR")

if configured_data_dir:
    project_path = Path(configured_data_dir).expanduser().resolve()
else:
    project_path = (Path.cwd() / "data").resolve()

if not project_path.exists():
    raise FileNotFoundError(
        f"Data directory not found: {project_path}. "
        "Create ./data or define WEBMINING_DATA_DIR."
    )

os.chdir(project_path)

print("Dossier courant :")
print(os.getcwd())

print("\nFichiers CSV disponibles dans le dossier :")
for f in os.listdir():
    if f.endswith(".csv"):
        print("-", f)


Dossier courant :
C:\Users\asus\Documents\dataset_WebMining

Fichiers CSV disponibles dans le dossier :
- aijobs_raw.csv
- final_data_ai_jobs_clean_2020_2026.csv
- final_data_ai_jobs_merged_2020_2026.csv
- final_data_ai_jobs_skills_exploded.csv
- global_ai_jobs_dataset.csv
- huggingface_global_2023_data_jobs_skills_filtered.csv
- jobs_data_ai_clean.csv
- kaggle_global_2025_data_science_jobs_skills_filtered.csv
- linkedin_datastax_2023-12-05_to_2024-04-20_data_ai_filtered.csv
- linkedin_us_uk_canada_australia_2024-01-12_to_2024-01-17_data_ai_skills_filtered.csv
- remoteok_raw.csv


## 3. Connexion à MongoDB Local

MongoDB Compass doit être ouvert et connecté à :

```text
mongodb://localhost:27017/
```

Le code suivant vérifie que Python peut se connecter à MongoDB.

In [5]:
client = MongoClient("mongodb://localhost:27017/")

print("Connexion réussie à MongoDB.")
print("Bases existantes :")
print(client.list_database_names())

Connexion réussie à MongoDB.
Bases existantes :
['admin', 'config', 'decisionnel', 'local']


## 4. Création de la base et des collections

La base et les collections seront créées automatiquement lors de l’insertion des données.

In [6]:
db = client["data_ai_jobs_db"]

jobs_collection = db["jobs_clean"]
skills_collection = db["skills_exploded"]

print("Base préparée : data_ai_jobs_db")
print("Collections préparées : jobs_clean, skills_exploded")

Base préparée : data_ai_jobs_db
Collections préparées : jobs_clean, skills_exploded


## 5. Chargement du dataset des offres

On utilise de préférence la version réduite issue du Data Quality Framework, car elle supprime les colonnes très incomplètes comme `description`, `job_url`, `company_name` et `job_type`.

Si le fichier réduit n’existe pas, on utilise la version complète.

In [7]:
reduced_file = "final_data_ai_jobs_clean_reduced_2020_2026.csv"
full_file = "final_data_ai_jobs_clean_2020_2026.csv"

if os.path.exists(reduced_file):
    jobs_file = reduced_file
else:
    jobs_file = full_file

print("Fichier utilisé pour jobs_clean :", jobs_file)

df_jobs = pd.read_csv(jobs_file)

print("Shape df_jobs :", df_jobs.shape)
df_jobs.head()

Fichier utilisé pour jobs_clean : final_data_ai_jobs_clean_2020_2026.csv
Shape df_jobs : (751801, 30)


,source,platform,job_id,job_title,company_name,country,city,date_posted,year,month,...,salary,salary_currency,job_url,original_file,country_clean,skills_clean,skills_original,remote_status_clean,remote_status_original,job_category_clean
0,global_ai_jobs_dataset,Synthetic / Global,1,AI Researcher,NaN,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,...,104015,USD,NaN,global_ai_jobs_dataset.csv,Canada,"python, computer vision, sql, nlp",Python;Computer Vision;SQL;NLP,Remote,Remote,Machine Learning / AI
1,global_ai_jobs_dataset,Synthetic / Global,2,MLOps Engineer,NaN,India,Tokyo,2020-04-12 00:00:00,2020,4.0,...,103263,USD,NaN,global_ai_jobs_dataset.csv,India,"nlp, pytorch, data analysis, computer vision",NLP;PyTorch;Data Analysis;Computer Vision,Remote,Remote,MLOps / Cloud Data
2,global_ai_jobs_dataset,Synthetic / Global,3,Data Analyst,NaN,United Kingdom,Bangalore,2023-01-31 00:00:00,2023,1.0,...,104190,USD,NaN,global_ai_jobs_dataset.csv,United Kingdom,"nlp, computer vision, python, pytorch",NLP;Computer Vision;Python;PyTorch,Hybrid,Hybrid,Data Analysis / BI
3,global_ai_jobs_dataset,Synthetic / Global,4,NLP Engineer,NaN,Brazil,Amsterdam,2022-09-30 00:00:00,2022,9.0,...,51440,USD,NaN,global_ai_jobs_dataset.csv,Brazil,"data analysis, statistics, tensorflow, python",Data Analysis;Statistics;TensorFlow;Python,Remote,Remote,Machine Learning / AI
4,global_ai_jobs_dataset,Synthetic / Global,5,AI Researcher,NaN,Netherlands,Amsterdam,2022-07-03 00:00:00,2022,7.0,...,58358,USD,NaN,global_ai_jobs_dataset.csv,Netherlands,"nlp, machine learning, pytorch, sql",NLP;Machine Learning;PyTorch;SQL,Hybrid,Hybrid,Machine Learning / AI


## 6. Préparation du dataset des offres pour MongoDB

MongoDB stocke les données sous forme de documents JSON/BSON. On prépare donc les colonnes :

- conversion de `date_posted` en texte lisible ;
- transformation de `skills` en liste ;
- remplacement des valeurs manquantes par `None`.

In [9]:
jobs_mongo = df_jobs.copy()

if "date_posted" in jobs_mongo.columns:
    jobs_mongo["date_posted"] = pd.to_datetime(
        jobs_mongo["date_posted"], errors="coerce"
    ).astype(str)

def skills_to_list(value):
    if pd.isna(value):
        return []
    return [s.strip().lower() for s in str(value).split(",") if s.strip()]

if "skills" in jobs_mongo.columns:
    jobs_mongo["skills_list"] = jobs_mongo["skills"].apply(skills_to_list)

jobs_mongo = jobs_mongo.replace({np.nan: None})

print("Préparation terminée.")
print("Shape jobs_mongo :", jobs_mongo.shape)
jobs_mongo.head().T

Préparation terminée.
Shape jobs_mongo : (751801, 31)


,0,1,2,3,4
source,global_ai_jobs_dataset,global_ai_jobs_dataset,global_ai_jobs_dataset,global_ai_jobs_dataset,global_ai_jobs_dataset
platform,Synthetic / Global,Synthetic / Global,Synthetic / Global,Synthetic / Global,Synthetic / Global
job_id,1,2,3,4,5
job_title,AI Researcher,MLOps Engineer,Data Analyst,NLP Engineer,AI Researcher
company_name,None,None,None,None,None
country,Canada,India,United Kingdom,Brazil,Netherlands
city,Berlin,Tokyo,Bangalore,Amsterdam,Amsterdam
date_posted,2021-04-01 00:00:00,2020-04-12 00:00:00,2023-01-31 00:00:00,2022-09-30 00:00:00,2022-07-03 00:00:00
year,2021,2020,2023,2022,2022
month,4.0,4.0,1.0,9.0,7.0


## 7. Insertion de `jobs_clean` dans MongoDB

L’insertion se fait par lots pour éviter de bloquer la mémoire.

> Remarque : si la collection existe déjà, elle sera vidée avant la nouvelle insertion.

In [11]:
jobs_collection.delete_many({})

records_jobs = jobs_mongo.to_dict(orient="records")

batch_size = 5000

for i in range(0, len(records_jobs), batch_size):
    batch = records_jobs[i:i + batch_size]
    if batch:
        jobs_collection.insert_many(batch)
    print(f"Insertion jobs : {i} à {i + len(batch)}")

print("Insertion jobs terminée.")
print("Nombre de documents dans jobs_clean :", jobs_collection.count_documents({}))

Insertion jobs : 0 à 5000
Insertion jobs : 5000 à 10000
Insertion jobs : 10000 à 15000
Insertion jobs : 15000 à 20000
Insertion jobs : 20000 à 25000
Insertion jobs : 25000 à 30000
Insertion jobs : 30000 à 35000
Insertion jobs : 35000 à 40000
Insertion jobs : 40000 à 45000
Insertion jobs : 45000 à 50000
Insertion jobs : 50000 à 55000
Insertion jobs : 55000 à 60000
Insertion jobs : 60000 à 65000
Insertion jobs : 65000 à 70000
Insertion jobs : 70000 à 75000
Insertion jobs : 75000 à 80000
Insertion jobs : 80000 à 85000
Insertion jobs : 85000 à 90000
Insertion jobs : 90000 à 95000
Insertion jobs : 95000 à 100000
Insertion jobs : 100000 à 105000
Insertion jobs : 105000 à 110000
Insertion jobs : 110000 à 115000
Insertion jobs : 115000 à 120000
Insertion jobs : 120000 à 125000
Insertion jobs : 125000 à 130000
Insertion jobs : 130000 à 135000
Insertion jobs : 135000 à 140000
Insertion jobs : 140000 à 145000
Insertion jobs : 145000 à 150000
Insertion jobs : 150000 à 155000
Insertion jobs : 15500

## 8. Vérification rapide de la collection `jobs_clean`

In [13]:
print("Nombre total de documents jobs_clean :", jobs_collection.count_documents({}))

print("Exemple de document :")
example_job = jobs_collection.find_one()
example_job

Nombre total de documents jobs_clean : 751801
Exemple de document :


{'_id': ObjectId('6a0a04c9d4c062cdd5bd5923'),
 'source': 'global_ai_jobs_dataset',
 'platform': 'Synthetic / Global',
 'job_id': 1,
 'job_title': 'AI Researcher',
 'company_name': None,
 'country': 'Canada',
 'city': 'Berlin',
 'date_posted': '2021-04-01 00:00:00',
 'year': 2021,
 'month': 4.0,
 'year_month': '2021-04',
 'job_type': None,
 'remote_status': 'Remote',
 'experience_level': 'Senior',
 'education_required': 'PhD',
 'industry': 'Manufacturing',
 'category': 'Generative AI',
 'description': None,
 'skills': 'python, computer vision, sql, nlp',
 'tools_used': 'Kubernetes;PyTorch;AWS',
 'salary': 104015,
 'salary_currency': 'USD',
 'job_url': None,
 'original_file': 'global_ai_jobs_dataset.csv',
 'country_clean': 'Canada',
 'skills_clean': 'python, computer vision, sql, nlp',
 'skills_original': 'Python;Computer Vision;SQL;NLP',
 'remote_status_clean': 'Remote',
 'remote_status_original': 'Remote',
 'job_category_clean': 'Machine Learning / AI',
 'skills_list': ['python', 'comp

## 9. Chargement du dataset des compétences explosées

Le fichier `final_data_ai_jobs_skills_exploded.csv` contient une ligne par compétence et par offre.

In [14]:
skills_file = "final_data_ai_jobs_skills_exploded.csv"

if not os.path.exists(skills_file):
    raise FileNotFoundError(f"Le fichier {skills_file} est introuvable dans le dossier courant.")

df_skills = pd.read_csv(skills_file)

print("Fichier utilisé pour skills_exploded :", skills_file)
print("Shape df_skills :", df_skills.shape)
df_skills.head()

Fichier utilisé pour skills_exploded : final_data_ai_jobs_skills_exploded.csv
Shape df_skills : (3991435, 15)


,job_id,job_title,company_name,country,city,date_posted,year,month,year_month,remote_status,category,platform,source,original_file,skill
0,1,AI Researcher,NaN,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,python
1,1,AI Researcher,NaN,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,computer vision
2,1,AI Researcher,NaN,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,sql
3,1,AI Researcher,NaN,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,nlp
4,2,MLOps Engineer,NaN,India,Tokyo,2020-04-12 00:00:00,2020,4.0,2020-04,Remote,Recommendation Systems,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,nlp


## 10. Préparation du dataset skills pour MongoDB

In [15]:
skills_mongo = df_skills.copy()

if "date_posted" in skills_mongo.columns:
    skills_mongo["date_posted"] = pd.to_datetime(
        skills_mongo["date_posted"], errors="coerce"
    ).astype(str)

if "skill" in skills_mongo.columns:
    skills_mongo["skill"] = skills_mongo["skill"].astype(str).str.strip().str.lower()

skills_mongo = skills_mongo.replace({np.nan: None})

print("Préparation skills terminée.")
print("Shape skills_mongo :", skills_mongo.shape)
skills_mongo.head()

Préparation skills terminée.
Shape skills_mongo : (3991435, 15)


,job_id,job_title,company_name,country,city,date_posted,year,month,year_month,remote_status,category,platform,source,original_file,skill
0,1,AI Researcher,None,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,python
1,1,AI Researcher,None,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,computer vision
2,1,AI Researcher,None,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,sql
3,1,AI Researcher,None,Canada,Berlin,2021-04-01 00:00:00,2021,4.0,2021-04,Remote,Generative AI,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,nlp
4,2,MLOps Engineer,None,India,Tokyo,2020-04-12 00:00:00,2020,4.0,2020-04,Remote,Recommendation Systems,Synthetic / Global,global_ai_jobs_dataset,global_ai_jobs_dataset.csv,nlp


## 11. Insertion de `skills_exploded` dans MongoDB

Ce fichier contient plusieurs millions de lignes. L’insertion peut prendre du temps.

> Si ton ordinateur est lent, tu peux augmenter ou réduire `batch_size`.

In [17]:
skills_collection.delete_many({})

records_skills = skills_mongo.to_dict(orient="records")

batch_size = 10000

for i in range(0, len(records_skills), batch_size):
    batch = records_skills[i:i + batch_size]
    if batch:
        skills_collection.insert_many(batch)
    print(f"Insertion skills : {i} à {i + len(batch)}")

print("Insertion skills terminée.")
print("Nombre de documents dans skills_exploded :", skills_collection.count_documents({}))

Insertion skills : 0 à 10000
Insertion skills : 10000 à 20000
Insertion skills : 20000 à 30000
Insertion skills : 30000 à 40000
Insertion skills : 40000 à 50000
Insertion skills : 50000 à 60000
Insertion skills : 60000 à 70000
Insertion skills : 70000 à 80000
Insertion skills : 80000 à 90000
Insertion skills : 90000 à 100000
Insertion skills : 100000 à 110000
Insertion skills : 110000 à 120000
Insertion skills : 120000 à 130000
Insertion skills : 130000 à 140000
Insertion skills : 140000 à 150000
Insertion skills : 150000 à 160000
Insertion skills : 160000 à 170000
Insertion skills : 170000 à 180000
Insertion skills : 180000 à 190000
Insertion skills : 190000 à 200000
Insertion skills : 200000 à 210000
Insertion skills : 210000 à 220000
Insertion skills : 220000 à 230000
Insertion skills : 230000 à 240000
Insertion skills : 240000 à 250000
Insertion skills : 250000 à 260000
Insertion skills : 260000 à 270000
Insertion skills : 270000 à 280000
Insertion skills : 280000 à 290000
Insertio

## 12. Vérification dans MongoDB

Après exécution des insertions, retourne dans MongoDB Compass. Tu dois voir :

```text
data_ai_jobs_db
    jobs_clean
    skills_exploded
```

Les volumes attendus sont environ :

```text
jobs_clean : 751 801 documents
skills_exploded : 3 991 435 documents
```

In [20]:
print("Bases MongoDB disponibles :")
print(client.list_database_names())

print("Collections dans data_ai_jobs_db :")
print(db.list_collection_names())

print("Nombre de documents jobs_clean :", jobs_collection.count_documents({}))
print("Nombre de documents skills_exploded :", skills_collection.count_documents({}))

Bases MongoDB disponibles :
['admin', 'config', 'data_ai_jobs_db', 'decisionnel', 'local']
Collections dans data_ai_jobs_db :
['skills_exploded', 'jobs_clean']
Nombre de documents jobs_clean : 751801
Nombre de documents skills_exploded : 3991435


# 13. Requêtes MongoDB principales

Ces requêtes servent à prouver que la base NoSQL est exploitable.

## 13.1 Nombre total d'offres

In [21]:
total_jobs = jobs_collection.count_documents({})
print("Nombre total d'offres :", total_jobs)

Nombre total d'offres : 751801


## 13.2 Nombre d'offres au Maroc

In [22]:
morocco_jobs = jobs_collection.count_documents({"country": "Morocco"})
print("Nombre d'offres au Maroc :", morocco_jobs)

Nombre d'offres au Maroc : 1058


## 13.3 Nombre d'offres Remote

In [23]:
remote_jobs = jobs_collection.count_documents({"remote_status": "Remote"})
print("Nombre d'offres Remote :", remote_jobs)

Nombre d'offres Remote : 100902


## 13.4 Nombre d'offres contenant Python

In [24]:
python_jobs = jobs_collection.count_documents({"skills_list": "python"})
print("Nombre d'offres contenant Python :", python_jobs)

Nombre d'offres contenant Python : 416081


## 13.5 Répartition des offres par année

In [25]:
jobs_by_year = list(jobs_collection.aggregate([
    {"$group": {"_id": "$year", "number_of_jobs": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
]))

pd.DataFrame(jobs_by_year).rename(columns={"_id": "year"})

,year,number_of_jobs
0,2020,20010
1,2021,20175
2,2022,19872
3,2023,641784
4,2024,27642
5,2025,20985
6,2026,1333


## 13.6 Top 20 pays

In [26]:
top_countries = list(jobs_collection.aggregate([
    {"$group": {"_id": "$country", "number_of_jobs": {"$sum": 1}}},
    {"$sort": {"number_of_jobs": -1}},
    {"$limit": 20}
]))

pd.DataFrame(top_countries).rename(columns={"_id": "country"})

,country,number_of_jobs
0,United States,199743
1,India,56012
2,United Kingdom,46123
3,Germany,33488
4,France,32273
5,Singapore,31216
6,Netherlands,28268
7,Canada,25601
8,Australia,21516
9,Spain,20278


## 13.7 Top sources

In [27]:
top_sources = list(jobs_collection.aggregate([
    {"$group": {"_id": "$original_file", "number_of_jobs": {"$sum": 1}}},
    {"$sort": {"number_of_jobs": -1}}
]))

pd.DataFrame(top_sources).rename(columns={"_id": "original_file"})

,original_file,number_of_jobs
0,huggingface_global_2023_data_jobs_skills_filte...,622067
1,global_ai_jobs_dataset.csv,120000
2,linkedin_us_uk_canada_australia_2024-01-12_to_...,5732
3,linkedin_datastax_2023-12-05_to_2024-04-20_dat...,1777
4,jobs_raw.json,960
5,kaggle_global_2025_data_science_jobs_skills_fi...,941
6,jobs_data_ai_clean.csv,274
7,aijobs_raw.csv,50


## 13.8 Top 30 skills

In [28]:
top_skills = list(skills_collection.aggregate([
    {"$group": {"_id": "$skill", "number_of_mentions": {"$sum": 1}}},
    {"$sort": {"number_of_mentions": -1}},
    {"$limit": 30}
]))

pd.DataFrame(top_skills).rename(columns={"_id": "skill"})

,skill,number_of_mentions
0,sql,423016
1,python,416081
2,aws,133924
3,r,131196
4,tableau,126609
5,excel,124313
6,azure,123904
7,spark,112002
8,power bi,97363
9,tensorflow,81587


## 13.9 Top skills au Maroc

In [29]:
top_skills_morocco = list(skills_collection.aggregate([
    {"$match": {"country": "Morocco"}},
    {"$group": {"_id": "$skill", "number_of_mentions": {"$sum": 1}}},
    {"$sort": {"number_of_mentions": -1}},
    {"$limit": 30}
]))

pd.DataFrame(top_skills_morocco).rename(columns={"_id": "skill"})

,skill,number_of_mentions
0,python,515
1,sql,510
2,spark,265
3,r,211
4,azure,209
5,tableau,193
6,power bi,191
7,java,181
8,hadoop,178
9,excel,172


## 13.10 Offres par remote_status

In [30]:
remote_distribution = list(jobs_collection.aggregate([
    {"$group": {"_id": "$remote_status", "number_of_jobs": {"$sum": 1}}},
    {"$sort": {"number_of_jobs": -1}}
]))

pd.DataFrame(remote_distribution).rename(columns={"_id": "remote_status"})

,remote_status,number_of_jobs
0,On-site,608492
1,Remote,100902
2,Hybrid,40077
3,Not specified,2330


# 14. Création d'index MongoDB

Les index améliorent la rapidité des requêtes fréquentes.

In [31]:
jobs_collection.create_index("country")
jobs_collection.create_index("year")
jobs_collection.create_index("remote_status")
jobs_collection.create_index("original_file")
jobs_collection.create_index("skills_list")

skills_collection.create_index("skill")
skills_collection.create_index("country")
skills_collection.create_index("year")

print("Index créés avec succès.")

Index créés avec succès.


# 15. Exemples de requêtes filtrées

## 15.1 Afficher quelques offres au Maroc

In [32]:
examples_morocco = list(jobs_collection.find(
    {"country": "Morocco"},
    {"_id": 0, "job_title": 1, "country": 1, "year": 1, "remote_status": 1, "skills_list": 1}
).limit(5))

pd.DataFrame(examples_morocco)

,job_title,country,year,remote_status,skills_list
0,JAVA JEE Developer (M/F),Morocco,2026,On-site,"[ai, angular, aws, azure, big data, cassandra,..."
1,Engineer Software,Morocco,2026,On-site,"[.net, ai, aws, azure, c#, ci, cd, cloud, dock..."
2,AWS Solutions Architect (M/F),Morocco,2026,Remote,"[ai, aws, big data, cloud, data analytics, doc..."
3,DevOps Engineer (M/F),Morocco,2026,On-site,"[ai, angular, ansible, aws, azure, ci, cd, clo..."
4,Senior DevOps Engineer,Morocco,2026,Remote,"[ansible, aws, bash, ci, cd, cloud, cybersecur..."


## 15.2 Afficher quelques offres contenant Python

In [33]:
examples_python = list(jobs_collection.find(
    {"skills_list": "python"},
    {"_id": 0, "job_title": 1, "country": 1, "year": 1, "remote_status": 1, "skills_list": 1}
).limit(5))

pd.DataFrame(examples_python)

,job_title,country,year,remote_status,skills_list
0,AI Researcher,Canada,2021,Remote,"[python, computer vision, sql, nlp]"
1,Data Analyst,United Kingdom,2023,Hybrid,"[nlp, computer vision, python, pytorch]"
2,NLP Engineer,Brazil,2022,Remote,"[data analysis, statistics, tensorflow, python]"
3,AI Engineer,Singapore,2021,Hybrid,"[statistics, python, tensorflow, machine learn..."
4,Data Analyst,Germany,2022,Hybrid,"[python, deep learning, sql, pytorch]"


# 16. Captures à faire dans MongoDB Compass

Pour le rapport, il faut faire des captures de :

1. La base `data_ai_jobs_db` dans MongoDB Compass.
2. La collection `jobs_clean` avec le nombre de documents.
3. La collection `skills_exploded` avec le nombre de documents.
4. Un exemple de document dans `jobs_clean`.
5. Un filtre sur `country: "Morocco"`.
6. Un filtre sur une compétence comme `skill: "python"` dans `skills_exploded`.

Ces captures prouvent que la base NoSQL est bien créée et exploitable.

# 17. Conclusion

Dans ce notebook, nous avons stocké le dataset final dans MongoDB afin de répondre à l'exigence d'une architecture de stockage NoSQL.

Deux collections ont été créées :

- `jobs_clean` : contient les offres d’emploi nettoyées ;
- `skills_exploded` : contient les compétences séparées une par une.

Cette architecture permet de réaliser rapidement des requêtes sur les pays, les années, les compétences, les sources et le mode de travail. MongoDB est adapté à ce projet car les offres d'emploi sont semi-structurées et proviennent de sources hétérogènes.